# quantbench_paper_soxl_v1

Notebook-first evidence surface for adapting QuantBench (arXiv:2504.18600) to the SOXL harness.

The architecture logic lives in `src.soxl_quant.app.quantbench_screen`, `src.soxl_quant.app.quantbench_supervised`, and `src.soxl_quant.app.quantbench_ensemble`; this notebook only executes that logic and records the resulting evidence.


In [1]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

def find_repo_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for path in [current, *current.parents]:
        if (path / "AGENTS.md").exists() and (path / "SPEC.md").exists():
            return path
    raise RuntimeError("repo root not found")

ROOT = find_repo_root()
CARD = ROOT / "runtime" / "model_selection" / "card" / "quantbench_paper_soxl_v1"
CARD.mkdir(parents=True, exist_ok=True)

import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.quantbench_screen import QuantBenchGate, build_quantbench_screen_report
from src.soxl_quant.app.quantbench_supervised import build_quantbench_supervised_report
from src.soxl_quant.app.quantbench_ensemble import build_quantbench_supervised_ensemble_report
from src.soxl_quant.data import build_dataset, init_qlib
from src.soxl_quant.settings import QLIB_DATA_PATH

print("root=.")
print(f"card={CARD.relative_to(ROOT)}")

root=.
card=runtime/model_selection/card/quantbench_paper_soxl_v1


## Execute Src-Owned QuantBench Screen

The strict gate encodes the user target: full test-proxy MaxDD <= 20%, positive proxy return, strong 2022 benchmark-relative defense, nonzero stress exposure so the strategy retains chips, and at least 50% capture of 2023-2024 buy-and-hold upside.

In [2]:
gate = QuantBenchGate(
    max_drawdown_pct=20.0,
    min_stress_relative_return=30.0,
    min_stress_avg_pos=0.10,
    max_stress_avg_pos=0.45,
    min_bull_capture=0.50,
    min_proxy_return=0.0,
)
report = build_quantbench_screen_report(root=ROOT, card_dir=CARD, gate=gate)
print(json.dumps({
    "decision": report["decision"],
    "records": report["screen"]["records"],
    "eligible_profiles": report["screen"]["eligible_profiles"],
}, indent=2))

{
  "decision": "no_current_model_satisfies_quantbench_soxl_20dd_gate",
  "records": 26,
  "eligible_profiles": []
}


## Execute Supervised QuantBench Models

This section trains the paper-style supervised model/objective families through `src.soxl_quant.app.quantbench_supervised` on the existing SOXL feature architecture, including macro-risk and relation packs.

In [3]:
feature_packs = (
    "vix_term",
    "academic_signals",
    "macro_signals",
    "rulebase_signals",
    "transition_signals",
    "soxx_signals",
    "soxx_spread_oscillation",
)

init_qlib(QLIB_DATA_PATH)
dataset = build_dataset(feature_packs=feature_packs, profile_name="quantbench_paper_soxl_v1")
features = dataset.handler.feature_frame
close = dataset.handler.price_frame["price"]

supervised_report = build_quantbench_supervised_report(
    features,
    close,
    card_dir=CARD,
    root=ROOT,
    horizon=20,
    feature_packs=feature_packs,
    gate=gate,
)
print(json.dumps({
    "decision": supervised_report["decision"],
    "records": supervised_report["screen"]["records"],
    "eligible_profiles": supervised_report["screen"]["eligible_profiles"],
    "feature_count": supervised_report["data"]["feature_count"],
}, indent=2))

[56358:MainThread](2026-06-07 04:41:56,207) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[56358:MainThread](2026-06-07 04:41:56,556) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[56358:MainThread](2026-06-07 04:41:56,557) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/.worktrees/quantbench-paper-soxl-v1/data/qlib/soxl')}


[56358:MainThread](2026-06-07 04:41:58,156) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[56358:MainThread](2026-06-07 04:41:58,156) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[56358:MainThread](2026-06-07 04:41:58,157) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Init data Done


{
  "decision": "no_supervised_model_satisfies_quantbench_soxl_20dd_gate",
  "records": 65,
  "eligible_profiles": [],
  "feature_count": 113
}


## Execute Supervised Path Ensemble

This section uses `src.soxl_quant.app.quantbench_ensemble` to turn supervised model predictions into reusable daily position paths, select among defensive/offensive paths with macro-risk regimes, and apply configured hazard, realized-volatility budget, and account-drawdown governors.


In [4]:
ensemble_report = build_quantbench_supervised_ensemble_report(
    features,
    close,
    card_dir=CARD,
    root=ROOT,
    horizon=20,
    feature_packs=feature_packs,
    gate=gate,
)
print(json.dumps({
    "decision": ensemble_report["decision"],
    "records": ensemble_report["screen"]["records"],
    "eligible_profiles": ensemble_report["screen"]["eligible_profiles"],
    "best_profile": ensemble_report["screen"]["best_rows"][0]["profile"] if ensemble_report["screen"]["best_rows"] else None,
}, indent=2))


{
  "decision": "eligible_ensemble_candidates_found",
  "records": 27,
  "eligible_profiles": [
    "quantbench_ensemble_sequence_fragile_panic_budget_v1"
  ],
  "best_profile": "quantbench_ensemble_sequence_fragile_panic_budget_v1"
}


## Supervised Model / Objective Screen

In [5]:
supervised_specs_df = pd.read_csv(CARD / "supervised_model_specs.csv")
supervised_metrics_df = pd.read_csv(CARD / "supervised_objective_metrics.csv")
supervised_screen_df = pd.read_csv(CARD / "supervised_candidate_screen.csv")
print({
    "eligible": int(supervised_screen_df["eligible"].sum()) if not supervised_screen_df.empty else 0,
    "rows": len(supervised_screen_df),
})
display(supervised_specs_df)
display(supervised_metrics_df)
display(supervised_screen_df.head(20))

{'eligible': 0, 'rows': 65}


,name,paper_family,objective,target_kind,estimator
0,logistic_clf,linear_classifier,clf,binary,LogisticRegression
1,ridge_ic,linear_regression,ic,return,Ridge
2,ridge_mse,linear_regression,mse,return,Ridge
3,random_forest_clf,tree,clf,binary,RandomForestClassifier
4,hist_gradient_mse,boosted_tree,mse,return,HistGradientBoostingRegressor
5,extra_trees_rank,tree_ensemble,ranking,return,ExtraTreesRegressor
6,mlp_mse,neural_tabular,mse,return,MLPRegressor
7,sequence_lag_ridge_mse,sequence_temporal,mse,return,LaggedRidge
8,lstm_mse,sequence_recurrent,mse,return,TorchLSTM
9,tcn_mse,sequence_convolutional,mse,return,TorchTemporalConv


,model_name,paper_family,objective,rows,clf_accuracy,ic,mse,ranking_spread,utility_return,base_model_name,fold_count,min_train_rows,refit_step_rows,train_lookback_rows,folds
0,logistic_clf,linear_classifier,clf,1594.0,0.471142,0.084410,0.267675,0.007406,0.018179,NaN,NaN,NaN,NaN,NaN,NaN
1,ridge_ic,linear_regression,ic,1594.0,0.444793,-0.110690,0.317109,-0.007968,0.009132,NaN,NaN,NaN,NaN,NaN,NaN
2,ridge_mse,linear_regression,mse,1594.0,0.444793,-0.110690,0.317109,-0.007968,0.009132,NaN,NaN,NaN,NaN,NaN,NaN
3,random_forest_clf,tree,clf,1594.0,0.500000,-0.032808,0.104747,-0.074851,0.008161,NaN,NaN,NaN,NaN,NaN,NaN
4,hist_gradient_mse,boosted_tree,mse,1594.0,0.456713,0.049200,0.111894,0.054097,0.010012,NaN,NaN,NaN,NaN,NaN,NaN
5,extra_trees_rank,tree_ensemble,ranking,1594.0,0.493726,-0.017903,0.099501,-0.003517,0.007681,NaN,NaN,NaN,NaN,NaN,NaN
6,mlp_mse,neural_tabular,mse,1594.0,0.484944,-0.012545,0.274140,0.003873,0.032648,NaN,NaN,NaN,NaN,NaN,NaN
7,sequence_lag_ridge_mse,sequence_temporal,mse,1594.0,0.425972,-0.140640,0.525640,-0.060645,0.008506,NaN,NaN,NaN,NaN,NaN,NaN
8,lstm_mse,sequence_recurrent,mse,1594.0,0.565245,0.112085,0.102754,0.072016,0.012941,NaN,NaN,NaN,NaN,NaN,NaN
9,tcn_mse,sequence_convolutional,mse,1594.0,0.507528,0.044354,0.127643,0.039506,0.014499,NaN,NaN,NaN,NaN,NaN,NaN


,profile,source_path,eligible,score,failed_checks,proxy_return,proxy_max_dd,stress_relative_return,stress_max_dd,stress_avg_pos,stress_window_avg_pos,stress_regime_avg_pos,bull_return,bull_buy_and_hold_return,bull_capture
0,quantbench_supervised_mlp_mse,generated:quantbench_supervised_mlp_mse,False,1353.355757,"['max_drawdown', 'stress_avg_pos', 'stress_max...",1498.768596,67.409087,63.676846,51.531111,0.498757,0.498757,NaN,231.176563,191.773509,1.205467
1,quantbench_supervised_market_graph_mse,generated:quantbench_supervised_market_graph_mse,False,946.380313,"['max_drawdown', 'stress_avg_pos', 'stress_max...",1089.886774,69.418784,81.323814,48.813575,0.498948,0.498948,NaN,216.406545,191.773509,1.128449
2,quantbench_supervised_ridge_ic,generated:quantbench_supervised_ridge_ic,False,724.062318,"['max_drawdown', 'stress_avg_pos', 'stress_max...",854.489316,68.178628,70.426229,50.351475,0.498452,0.498452,NaN,252.875013,191.773509,1.318613
3,quantbench_supervised_ridge_mse,generated:quantbench_supervised_ridge_mse,False,724.062318,"['max_drawdown', 'stress_avg_pos', 'stress_max...",854.489316,68.178628,70.426229,50.351475,0.498452,0.498452,NaN,252.875013,191.773509,1.318613
4,quantbench_supervised_logistic_clf,generated:quantbench_supervised_logistic_clf,False,611.587891,"['max_drawdown', 'stress_avg_pos', 'stress_max...",728.500036,66.516879,50.332270,53.015178,0.498300,0.498300,NaN,304.580636,191.773509,1.588231
5,quantbench_supervised_lstm_mse,generated:quantbench_supervised_lstm_mse,False,551.064039,"['max_drawdown', 'stress_avg_pos', 'stress_max...",762.057810,67.489228,47.703920,49.906906,0.499329,0.499329,NaN,136.656308,191.773509,0.712592
6,quantbench_supervised_tcn_mse,generated:quantbench_supervised_tcn_mse,False,481.375065,"['max_drawdown', 'stress_avg_pos', 'stress_max...",676.317141,71.860002,73.595280,47.546117,0.499240,0.499240,NaN,151.314389,191.773509,0.789027
7,quantbench_supervised_hist_gradient_mse,generated:quantbench_supervised_hist_gradient_mse,False,405.901101,"['max_drawdown', 'stress_avg_pos', 'stress_max...",681.761273,70.883673,40.881526,57.146925,0.498427,0.498427,NaN,89.736570,191.773509,0.467930
8,quantbench_supervised_random_forest_clf,generated:quantbench_supervised_random_forest_clf,False,367.879856,"['max_drawdown', 'stress_avg_pos', 'stress_max...",587.418900,65.661613,44.644049,53.546039,0.498186,0.498186,NaN,112.117238,191.773509,0.584634
9,quantbench_supervised_sequence_lag_ridge_mse,generated:quantbench_supervised_sequence_lag_r...,False,217.909927,"['max_drawdown', 'stress_avg_pos', 'stress_max...",451.380328,74.584312,53.293328,48.408489,0.498440,0.498440,NaN,137.259049,191.773509,0.715735


## Supervised Ensemble Screen


In [6]:
ensemble_screen_df = pd.read_csv(CARD / "ensemble_candidate_screen.csv")
ensemble_paths_df = pd.read_csv(CARD / "ensemble_path_library.csv")
ensemble_hazard_metrics = json.loads((CARD / "ensemble_hazard_metrics.json").read_text())
print({
    "eligible": int(ensemble_screen_df["eligible"].sum()) if not ensemble_screen_df.empty else 0,
    "rows": len(ensemble_screen_df),
    "hazard_mode": ensemble_hazard_metrics.get("mode"),
    "hazard_mean": round(float(ensemble_hazard_metrics.get("mean_hazard", 0.0)), 6),
    "hazard_fold_count": int(ensemble_hazard_metrics.get("fold_count", 0)),
})
display(ensemble_paths_df)
display(pd.DataFrame([ensemble_hazard_metrics]))
display(ensemble_screen_df.head(20))


{'eligible': 1, 'rows': 27, 'hazard_mode': 'rolling', 'hazard_mean': 0.186036, 'hazard_fold_count': 26}


,path,source,model_name,risk_overlay
0,quantbench_external_recovery_reentry_condition...,external_runtime_daily,quantbench_external_recovery_reentry_condition...,NaN
1,quantbench_external_risk_budget_crash_risk_fac...,external_runtime_daily,quantbench_external_risk_budget_crash_risk_fac...,NaN
2,quantbench_external_risk_budget_downside_vol_t...,external_runtime_daily,quantbench_external_risk_budget_downside_vol_t...,NaN
3,quantbench_external_risk_budget_residual_drawd...,external_runtime_daily,quantbench_external_risk_budget_residual_drawd...,NaN
4,quantbench_external_v19reg_v22_budget_usage_v1,external_runtime_daily,quantbench_external_v19reg_v22_budget_usage_v1,NaN
...,...,...,...,...
68,quantbench_supervised_tcn_mse,supervised_prediction,tcn_mse,NaN
69,quantbench_supervised_tcn_mse_macro_budget_bul...,supervised_prediction,tcn_mse,macro_budget_bull_guard_v1
70,quantbench_supervised_tcn_mse_macro_budget_def...,supervised_prediction,tcn_mse,macro_budget_defensive_v1
71,quantbench_supervised_tcn_mse_macro_budget_low...,supervised_prediction,tcn_mse,macro_budget_lowdd_v1


,drawdown_threshold,fold_count,folds,horizon,mean_hazard,min_train_rows,mode,refit_step_rows,test_rows,train_lookback_rows
0,-0.18,26,"[{'label_cutoff': '2019-12-02', 'predict_end':...",20,0.186036,756,rolling,63,1594,1260


,profile,source_path,eligible,score,failed_checks,proxy_return,proxy_max_dd,stress_relative_return,stress_max_dd,stress_avg_pos,stress_window_avg_pos,stress_regime_avg_pos,bull_return,bull_buy_and_hold_return,bull_capture
0,quantbench_ensemble_sequence_fragile_panic_bud...,generated:quantbench_ensemble_sequence_fragile...,True,390.506821,[],316.520167,19.742257,78.029331,14.365664,0.100494,0.086678,0.100494,143.688894,191.773509,0.749264
1,quantbench_ensemble_sequence_fragile_preshock_v1,generated:quantbench_ensemble_sequence_fragile...,False,384.100448,['max_drawdown'],333.061242,20.549701,79.068428,13.997291,0.100988,0.084343,0.100988,142.237607,191.773509,0.741696
2,quantbench_ensemble_sequence_tapered_reentry_v1,generated:quantbench_ensemble_sequence_tapered...,False,381.172610,['max_drawdown'],328.660811,20.549621,79.071045,13.996299,0.100988,0.084341,0.100988,145.056021,191.773509,0.756392
3,quantbench_ensemble_sequence_full_rebase_tail_v1,generated:quantbench_ensemble_sequence_full_re...,False,366.314774,['max_drawdown'],323.538434,24.011303,79.076552,13.994212,0.100988,0.084336,0.100988,152.929783,191.773509,0.797450
4,quantbench_ensemble_sequence_panic_chip_floor_v1,generated:quantbench_ensemble_sequence_panic_c...,False,286.520953,"['max_drawdown', 'stress_avg_pos']",265.421463,20.495085,78.426753,14.285860,0.099506,0.083704,0.099506,123.987470,191.773509,0.646531
5,quantbench_ensemble_market_graph_panic_chip_fl...,generated:quantbench_ensemble_market_graph_pan...,False,279.370021,"['max_drawdown', 'stress_avg_pos']",259.848123,20.528030,78.426781,14.285835,0.099506,0.083704,0.099506,121.214738,191.773509,0.632072
6,quantbench_ensemble_lstm_panic_chip_floor_v1,generated:quantbench_ensemble_lstm_panic_chip_...,False,262.722586,"['max_drawdown', 'stress_avg_pos']",249.300435,21.145315,78.426390,14.286197,0.099506,0.083705,0.099506,114.252943,191.773509,0.595770
7,quantbench_ensemble_tcn_panic_chip_floor_v1,generated:quantbench_ensemble_tcn_panic_chip_f...,False,243.920806,"['max_drawdown', 'stress_avg_pos']",232.770022,21.143833,78.427679,14.285000,0.099506,0.083703,0.099506,109.883226,191.773509,0.572984
8,quantbench_ensemble_external_crash_v19v22_v1,generated:quantbench_ensemble_external_crash_v...,False,149.830463,"['stress_avg_pos', 'bull_capture']",166.842013,17.512531,76.164217,12.382956,0.063911,0.067905,0.063911,32.360549,191.773509,0.168744
9,quantbench_ensemble_guarded_rf_logistic_v1,generated:quantbench_ensemble_guarded_rf_logis...,False,111.035327,"['max_drawdown', 'stress_avg_pos', 'stress_max...",250.816294,47.145779,54.166601,32.877233,0.080000,0.163948,0.080000,104.776506,191.773509,0.546355


## Paper Requirement Matrix

QuantBench is a full benchmark architecture: data preparation, factor mining, modeling, portfolio/execution objectives, and task-aware evaluation. The requirement matrix is emitted by src and persisted as a sidecar CSV.

In [7]:
requirements_df = pd.read_csv(CARD / "paper_requirement_matrix.csv")
display(requirements_df)

,key,paper_basis,soxl_mapping,next_artifact
0,data_preparation,"Unified data preparation with breadth, depth, ...","SOXL/SOXX/VIX daily panel, qlib binary store, ...",runtime/data or runtime/model_selection notebo...
1,factor_mining,Formula-based factor mining and learned factor...,"Feature pack diagnostics with IC, standalone b...",runtime/factor/card/<pack>/<pack>.ipynb plus f...
2,modeling_tree,Tree models can outperform deep nets on strong...,Add non-RL tabular baselines such as XGBoost/L...,runtime/model_selection/card/quantbench_paper_...
3,modeling_sequence,Temporal models such as LSTM are benchmarked a...,Existing PPO actor surfaces include MLP fallba...,network/profile matrix and smoke/full retrain ...
4,modeling_relational,Relational or graph models test whether market...,Single-SOXL setting maps this to SOXL/SOXX/VIX...,SOXX-relative and multi-asset relation feature...
5,training_objectives,"Classification, IC, MSE, ranking, utility, and...","Compare return-prediction labels, IC/ranking p...",objective-family screen with fixed-split and w...
6,rolling_continual_learning,Rolling updates mitigate alpha decay and distr...,Use existing walk-forward folds plus 3/6/12-mo...,runtime/model_selection rolling cadence notebo...
7,ensemble_robustness,Seed ensembles reduce low signal-to-noise over...,Aggregate low-correlated actors/signals only a...,ensemble candidate summary with pairwise corre...
8,evaluation_metrics,"Evaluation should include signal, portfolio, e...","Gate on IC/ICIR, return, Sharpe, MaxDD, turnov...",quantbench_screen summary.json plus candidate ...


## Model / Objective Coverage Ledger

This ledger is a coverage claim, not a performance claim. Missing families are explicit implementation tasks.

In [8]:
coverage_df = pd.read_csv(CARD / "paper_model_objective_coverage.csv")
display(coverage_df)

,family,paper_family,current_soxl_surface,coverage,next_step
0,tree_xgboost_lightgbm,XGBoost/tree model,sklearn tree/boosted supervised baselines impl...,implemented_partial,promote only if QuantBench gate passes; otherw...
1,sequence_lstm_gru,LSTM/temporal sequence,GRU actor exists; supervised TorchLSTM MSE bas...,implemented_partial,compare recurrent supervised signals with GRU ...
2,attention_tide_mixer,deep sequence / mixer style,attention and TiDE actor surfaces exist; super...,implemented_partial,compare TorchTCN and mixer-style signals under...
3,relational_rgcn,RGCN/relational model,SOXX-relative and multi-asset relation feature...,implemented_partial,promote only if MarketGraphRidge or graph-deri...
4,objectives_clf_ic_mse_ranking,CLF / IC / MSE / Ranking,common supervised target builder and CLF/IC/MS...,implemented,use objective metrics and risk-gate failures t...
5,utility_rl_execution,utility maximization / RL / execution,"PPO reward, risk budget, execution overlays exist",implemented_partial,use QuantBench gate for promotion selection
6,rolling_continual,3/6/12 month rolling updates,rolling supervised refit paths and rolling haz...,implemented_partial,keep cadence only if QuantBench gate improves;...
7,seed_ensemble,seed ensemble for overfit mitigation,low-correlation signal blend paths and macro p...,implemented_partial,keep only blends that improve the QuantBench g...


## Objective Family Mapping

This table is emitted by `src.soxl_quant.app.quantbench_objectives` and defines the shared SOXL target semantics for paper-style CLF, IC, MSE, ranking, utility, and RL comparisons.

In [9]:
objective_df = pd.read_csv(CARD / "paper_objective_families.csv")
display(objective_df)

,key,paper_basis,soxl_target,metric
0,clf,Classification loss predicts future up/down mo...,fwd_positive_h,clf_accuracy
1,ic,IC objective aligns predictions with future re...,fwd_return_h,ic
2,mse,MSE loss regresses future returns directly.,fwd_return_h,mse
3,ranking,Pairwise or listwise ranking objectives prefer...,relative ordering of fwd_return_h,ranking_spread
4,utility,Utility maximization should reflect trading ou...,position-weighted fwd_return_h with chip reten...,utility_return
5,rl,Non-differentiable simulation and execution fe...,PPO reward and execution gate metrics,portfolio_return_sharpe_maxdd


## Current Runtime Candidate Screen

The ranking scans current runtime summaries through the src-owned QuantBench gate. It does not retrain models in this notebook.

In [10]:
screen_df = pd.read_csv(CARD / "candidate_screen.csv")
print({"eligible": int(screen_df["eligible"].sum()) if not screen_df.empty else 0, "rows": len(screen_df)})
display(screen_df.head(20))

{'eligible': 0, 'rows': 26}


,profile,source_path,eligible,score,failed_checks,proxy_return,proxy_max_dd,stress_relative_return,stress_max_dd,stress_avg_pos,stress_window_avg_pos,stress_regime_avg_pos,bull_return,bull_buy_and_hold_return,bull_capture
0,balanced_rule_residual_v19_reg_v22_retest_v1,runtime/model/card/balanced_rule_residual_v19_...,False,1119.521092,"['max_drawdown', 'stress_max_drawdown']",1280.416932,59.637288,37.652465,58.299843,0.380930,0.380930,NaN,44.524050,55.654474,0.800008
1,balanced_rule_residual_v22_continued_retention_v1,runtime/model/card/balanced_rule_residual_v22_...,False,1015.381197,"['max_drawdown', 'stress_max_drawdown']",1128.935617,59.130395,53.736876,55.255626,0.321447,0.321447,NaN,60.791539,55.654474,1.092303
2,balanced_rule_residual_v22_release_v1,runtime/model/card/balanced_rule_residual_v22_...,False,1002.849540,"['max_drawdown', 'stress_max_drawdown']",1133.887381,60.350126,45.981620,38.845207,0.375263,0.375263,NaN,58.092719,55.654474,1.043810
3,balanced_rule_residual_v19_moe_v2,runtime/model/card/balanced_rule_residual_v19_...,False,610.398973,"['max_drawdown', 'stress_max_drawdown']",690.971221,49.008279,47.586050,45.149295,0.287632,0.287632,NaN,60.037161,55.654474,1.078748
4,state_conditioned_drawdown_rebound_reward_trai...,runtime/model/card/state_conditioned_drawdown_...,False,559.292883,"['max_drawdown', 'stress_max_drawdown']",619.708826,43.565118,52.906964,41.483544,0.258405,0.258405,NaN,56.176271,55.654474,1.009376
5,mode_conditioned_rebound_reward_train_v2,runtime/model/card/mode_conditioned_rebound_re...,False,426.778414,"['max_drawdown', 'stress_max_drawdown']",496.780712,44.234213,65.015154,43.909386,0.131381,0.131381,NaN,45.591811,55.654474,0.819194
6,drawdown_rebound_reward_train_v1,runtime/model/card/drawdown_rebound_reward_tra...,False,351.838655,"['max_drawdown', 'stress_max_drawdown']",392.580698,37.676486,55.990314,35.878546,0.185465,0.185465,NaN,52.300504,55.654474,0.939736
7,drawdown_rebound_reward_train_v1,runtime/model/card/drawdown_rebound_reward_tra...,False,-1032.939429,"['proxy_return', 'bull_capture']",-7.000281,9.875679,53.563568,9.875679,0.165855,0.165855,NaN,NaN,NaN,NaN
8,mode_conditioned_rebound_reward_train_v2,runtime/model/card/mode_conditioned_rebound_re...,False,-1054.613515,"['proxy_return', 'bull_capture']",-12.705282,15.900697,61.694557,15.900697,0.141487,0.141487,NaN,NaN,NaN,NaN
9,state_conditioned_drawdown_rebound_reward_trai...,runtime/model/card/state_conditioned_drawdown_...,False,-1174.741020,"['max_drawdown', 'proxy_return', 'stress_max_d...",-23.275008,29.147985,45.125927,29.147985,0.243813,0.243813,NaN,NaN,NaN,NaN


## Conclusion

This notebook executes the src-owned QuantBench-to-SOXL architecture screen against arXiv:2504.18600-style model families: runtime-summary screening, supervised CLF/IC/MSE/ranking baselines, temporal sequence baselines, recurrent and temporal-convolution supervised baselines, relation-aware `market_graph_mse` / `MarketGraphRidge` over SOXL-SOXX-VIX-spread nodes, non-leaking rolling supervised refits, low-correlation signal blends, repo-owned external runtime paths, and macro-risk path ensembles with hazard, volatility, fragile-market, tail-shock, account-drawdown, anchor-reset, chip-floor, and panic-budget governors.

Current run decision: supervised single-model candidates still fail the strict SOXL gate, but the src-owned path-ensemble screen now finds one eligible candidate: `quantbench_ensemble_sequence_fragile_panic_budget_v1`. It returns 316.52% on the proxy window with MaxDD 19.74%, 2022 relative defense 78.03%, 2022 stress MaxDD 14.37%, macro-stress-regime average position 10.05%, and 2023-2024 bull capture 74.93%. This satisfies the requested <20% drawdown constraint while retaining chips in extreme macro-risk rows and still participating in bull regimes.

The architecture change is not a notebook-only interpretation. `quantbench_screen` now treats macro stress-regime chip retention as the gate metric when `stress_regime_avg_pos` is present, while keeping backward compatibility for older summaries through the window-level `avg_pos` fallback. `quantbench_ensemble` now writes `stress_regime_avg_pos` and `stress_regime_rows` into each slice payload from the final selector output. The new default selector `quantbench_ensemble_sequence_fragile_panic_budget_v1` combines the sequence/tapered reentry path with fragile pre-shock caps and a true-panic budget cap: normal stress budget keeps roughly 12% target exposure, while confirmed tail-shock panic can cap to 10%.

Remaining risk: this is an ensemble/execution-layer candidate, not a clean supervised model winner. The current runtime-summary screen and supervised screen still report zero eligible rows under the same gate, so promotion should remain conditional on a follow-up walk-forward or out-of-sample replay. The next experiment should stress-test this candidate across alternative windows and then decide whether to promote it as a guarded execution overlay or use it as a teacher for the model/factor layer.

Primary artifacts: `ensemble_summary.json`, `ensemble_candidate_screen.csv`, `ensemble_payloads.json`, `quantbench_ensemble_sequence_fragile_panic_budget_v1_soxl_daily.csv`, `supervised_summary.json`, `supervised_candidate_screen.csv`, `summary.json`.
